# Home Credit Default Risk — Exploratory Data Analysis

This notebook explores `application_train.csv` before any modelling. It covers:
1. Dataset summary
2. Data-quality observations
3. Feature categorization
4. Business insights (with charts)

The same logic lives in `eda.py`, which writes `models/eda_summary.json` for the app.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.loader import load_application_train
sns.set_style('whitegrid')
df = load_application_train()
df.shape

## 1. Dataset summary

In [ ]:
print('Rows, Cols:', df.shape)
print('Numeric cols:', df.select_dtypes(include="number").shape[1])
print('Categorical cols:', df.select_dtypes(include="object").shape[1])
print('Default rate: {:.2%}'.format(df["TARGET"].mean()))
df['TARGET'].value_counts().rename({0:'Repaid',1:'Default'}).plot.bar(color=['#2e7d32','#c62828'], title='Target balance')
plt.show()

**Observation:** only ~8% of clients default. This class imbalance drives our metric choice (ROC-AUC / PR-AUC, not accuracy).

## 2. Data quality

In [ ]:
miss = (df.isna().mean()*100).round(1).sort_values(ascending=False)
miss[miss>0].head(15)

In [ ]:
# DAYS_EMPLOYED anomaly: 365243 is a sentinel for 'not employed' (pensioners)
print('Anomaly share: {:.1%}'.format((df['DAYS_EMPLOYED']==365243).mean()))
df['DAYS_EMPLOYED'].replace(365243, np.nan).div(-365.25).describe()

**Findings:** many columns are >40% null (building-info blocks); `EXT_SOURCE_1` is ~56% null but highly predictive; `DAYS_*` are negative; `DAYS_EMPLOYED` has an 18% sentinel anomaly.

## 3. Feature categorization

In [ ]:
groups = {
  'demographics': ['CODE_GENDER','DAYS_BIRTH','CNT_CHILDREN','NAME_FAMILY_STATUS'],
  'financials': ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE'],
  'employment/education': ['DAYS_EMPLOYED','OCCUPATION_TYPE','NAME_EDUCATION_TYPE'],
  'external scores': ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'],
  'housing': ['NAME_HOUSING_TYPE','FLAG_OWN_REALTY','FLAG_OWN_CAR'],
}
groups

## 4. Business insights

In [ ]:
# Insight 1: default rate by education
(df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().mul(100).sort_values()
   .plot.barh(color='#1565c0', title='Default rate (%) by education'))
plt.xlabel('Default rate (%)'); plt.show()

In [ ]:
# Insight 2: default rate by age band
age = (-df['DAYS_BIRTH']/365.25)
band = pd.cut(age, [20,30,40,50,60,100], labels=['20-30','30-40','40-50','50-60','60+'])
df.groupby(band)['TARGET'].mean().mul(100).plot.bar(color='#6a1b9a', title='Default rate (%) by age band')
plt.ylabel('Default rate (%)'); plt.show()

In [ ]:
# Insight 3: external scores are the strongest signals
df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','TARGET']].corr()['TARGET']

In [ ]:
# Insight 4: income vs credit, coloured by default
s = df.sample(5000, random_state=1)
plt.scatter(s['AMT_INCOME_TOTAL'].clip(upper=1e6), s['AMT_CREDIT'], c=s['TARGET'], cmap='coolwarm', alpha=0.4, s=8)
plt.xlabel('Income'); plt.ylabel('Credit'); plt.title('Income vs Credit (red=default)'); plt.show()

In [ ]:
# Insight 5: occupations with highest default rate
(df.groupby('OCCUPATION_TYPE')['TARGET'].agg(['mean','count'])
   .query('count>=1000').sort_values('mean', ascending=False).head(5)['mean'].mul(100))

### Key findings
1. ~8% default rate → heavy class imbalance.
2. Lower education defaults ~2x higher than higher education.
3. Younger applicants default more.
4. External credit scores are the strongest predictors.
5. Low-skill Laborers / Drivers have the highest occupational default rates.

Run `python notebooks/eda.py` to regenerate `models/eda_summary.json` used by the app.